# BTL03 – Chạy các notebook PyTorch trên Kaggle (GPU)

Notebook này chỉ clone repo, chuẩn bị dữ liệu rồi chạy lại đúng các notebook GPU `pytorch/**/*_pytorch.ipynb` bằng `nbconvert`; không chứa code mô hình riêng. Kết quả (notebook đã có output + `results/`) được nén thành một file zip để tải về.

**Trước khi chạy**
1. Trên máy: commit + push code mới nhất (notebook `git clone` repo `TysonNgo25/deep-learning-assignments`).
2. Kaggle → Create → New Notebook → File → Import Notebook → chọn file này.
3. Settings (cột phải):
   - Accelerator: **GPU T4 x2** (2 GPU, notebook chia 2 luồng song song; P100 chỉ có 1 GPU và PyTorch bản mới có thể không còn hỗ trợ đời GPU này).
   - Internet: **On** (để `git clone`, `pip install` và tải MNIST).
4. Add Input (cột phải → Add Input):
   - FER2013: tìm `nicolejyt/facialexpressionrecognition` (dataset đã dùng cho BTL02, có `fer2013.csv`).
   - Gender: dataset riêng tạo từ 4 file `data/gender/*.npy` (Datasets → New Dataset → kéo 4 file vào, để Private, ví dụ tên `btl03-gender`). Chỉ cần tạo một lần.
5. Chạy nền: **Save Version → Save & Run All (Commit)**. Đóng trình duyệt được; xong vào tab **Output** của version để tải `btl03_gpu_*.zip`.

**Sau khi tải về:** giải nén đè vào thư mục `BTL03 - Chuyển code từ keras sang pytorch deep learning/` trong repo (zip có sẵn đường dẫn `pytorch/...` và `results/...`), rồi `git status --short .` để xem file đổi.

| Luồng | GPU | Notebook (ước tính theo thời gian gốc GTX 1080) |
|---|---|---|
| 0 | cuda:0 | MNIST_CNN_LSTM (27 phút) → 1_MNIST_CNN (4) → 2_Human_Gender_CNN (3) |
| 1 | cuda:1 | tu_3_Emotion_Reg_CNN (23) → 3_Emotion_Reg_CNN (6) → tu_2_Human_Gender_CNN (3) → 1_segnet (12,5) |

Hai notebook dùng MNIST cùng nằm ở luồng 0, và MNIST được tải trước một lần, nên hai luồng không cùng tải một file.

In [ ]:
!nvidia-smi

In [ ]:
import glob, os, shutil, subprocess, sys, time

KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"     # chỉ để file zip và log ở đây (tab Output)
REPO = "/tmp/deep-learning-assignments"                # clone ra /tmp để Output không chứa cả repo
URL = "https://github.com/TysonNgo25/deep-learning-assignments.git"
BTL = "BTL03 - Chuyển code từ keras sang pytorch deep learning"

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, REPO], check=True)
os.chdir(os.path.join(REPO, BTL))
print("Thư mục làm việc:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"],
                     capture_output=True, text=True).stdout)

## 1. Dữ liệu

- Gender: tìm 4 file `.npy` ở bất kỳ đâu trong `/kaggle/input`, tạo liên kết vào `data/gender/` (không chép, đỡ tốn chỗ).
- FER2013: liên kết `fer2013.csv` vào `data/emotion/fer2013.csv` (`common.load_fer2013` đọc ở đây).
- MNIST: `torchvision` tải về `data/mnist/` (cần Internet On).
- SegNet, xe: có sẵn trong repo (`Yêu cầu/examples/5. Others/`).

In [ ]:
def lien_ket(ten_file, thu_muc_dich):
    os.makedirs(thu_muc_dich, exist_ok=True)
    dich = os.path.join(thu_muc_dich, ten_file)
    if os.path.exists(dich):
        return dich
    found = glob.glob(f"/kaggle/input/**/{ten_file}", recursive=True)
    if not found:
        raise FileNotFoundError(f"Không thấy {ten_file} trong /kaggle/input: kiểm tra Add Input")
    os.symlink(found[0], dich)
    return dich

for f in ["64_64_11938_4098_train_x_onehot.npy", "64_64_11938_4098_train_y_onehot.npy",
          "64_64_5968_4098_val_x_onehot.npy", "64_64_5968_4098_val_y_onehot.npy"]:
    p = lien_ket(f, "data/gender")
    print(p, os.path.getsize(p) // 2**20, "MB")
p = lien_ket("fer2013.csv", "data/emotion")
print(p, os.path.getsize(p) // 2**20, "MB")

In [ ]:
!pip install -q torchinfo
!python -c "import torch, torchvision, torchinfo, sklearn, skimage, seaborn, scipy, nbconvert; print('torch', torch.__version__, '| GPU:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

In [ ]:
# Tải MNIST một lần trước khi chạy song song
!python -c "import torchvision; [torchvision.datasets.MNIST('data/mnist', train=t, download=True) for t in (True, False)]; print('MNIST ok')"

## 2. Kiểm tra mô hình thật sự train trên GPU

Lần chạy 26–27/11 mất gần 12 giờ mà không xong: `summary()` cũ truyền `device="cpu"` cho torchinfo, torchinfo gọi `model.to("cpu")` nên mọi notebook train bằng CPU mà không báo lỗi. Đã sửa trong `common.py`; ô dưới kiểm lại trước khi chạy thật và dừng ngay nếu mô hình không còn ở GPU.

In [ ]:
kiem = '''
import sys; sys.path.insert(0, "pytorch")
from common import *
device = get_device()
assert device.type == "cuda", "Không thấy GPU: Settings -> Accelerator -> GPU T4 x2"
m = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(8 * 28 * 28, 10)).to(device)
summary(m, input_size=(1, 1, 28, 28), verbose=0)
assert next(m.parameters()).device.type == "cuda", "summary() đã kéo mô hình về CPU"
x, y = torch.randn(256, 1, 28, 28), torch.randint(0, 10, (256,))
h = fit(m, make_loader(x, y, 64), None, nn.CrossEntropyLoss(), torch.optim.SGD(m.parameters(), 0.01), 1)
print("OK: mô hình ở", next(m.parameters()).device)
'''
r = subprocess.run([sys.executable, "-c", kiem], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "Kiểm tra GPU thất bại, dừng lại"

## 3. Chạy các notebook

- Mỗi notebook chạy bằng `python -m nbconvert --execute --inplace`, output lưu vào chính file. Mỗi ô tối đa 2 giờ (`timeout=7200`): ô lâu nhất trên T4 dưới 1 giờ, nên quá 2 giờ là có vấn đề và notebook đó dừng với lỗi thay vì ăn hết 12 giờ. Notebook nào lỗi thì `nbconvert` dừng và không ghi đè file; log ở `/kaggle/working/log_<tên>.txt`.
- `CHON`: để `None` là chạy đủ 7 notebook GPU; muốn chạy lại vài cái thì ghi tên, ví dụ `CHON = ["1_segnet"]`.
- `2_obj_detection` nhỏ, đã chạy đủ trên CPU ở máy nên không có trong danh sách.
- Có 2 GPU thì chạy 2 luồng song song; 1 GPU thì chạy tuần tự.

In [ ]:
import torch

NOTEBOOK = {   # tên kết quả -> (thư mục trong pytorch/, tên file)
    "MNIST_CNN_LSTM":        ("3. CNN + LSTM", "MNIST_CNN_LSTM_pytorch.ipynb"),
    "1_MNIST_CNN":           ("2. CNN + MLP", "1_MNIST_CNN_pytorch.ipynb"),
    "2_Human_Gender_CNN":    ("2. CNN + MLP", "2_Human_Gender_CNN_pytorch.ipynb"),
    "tu_3_Emotion_Reg_CNN":  ("4. Training Utilities", "3_Emotion_Reg_CNN_pytorch.ipynb"),
    "3_Emotion_Reg_CNN":     ("2. CNN + MLP", "3_Emotion_Reg_CNN_pytorch.ipynb"),
    "tu_2_Human_Gender_CNN": ("4. Training Utilities", "2_Human_Gender_CNN_pytorch.ipynb"),
    "1_segnet":              ("5. Others", "1_segnet_pytorch.ipynb"),
}
CHON = None
streams = [["MNIST_CNN_LSTM", "1_MNIST_CNN", "2_Human_Gender_CNN"],
           ["tu_3_Emotion_Reg_CNN", "3_Emotion_Reg_CNN", "tu_2_Human_Gender_CNN", "1_segnet"]]
if CHON is not None:
    streams = [[n for n in s if n in CHON] for s in streams]
if torch.cuda.device_count() < 2:
    streams = [streams[0] + streams[1]]
streams = [s for s in streams if s]

BTL_DIR = os.getcwd()


def lenh(ten):
    thu_muc, file = NOTEBOOK[ten]
    log = os.path.join(WORK, f"log_{ten}.txt")
    return (f'cd "{BTL_DIR}/pytorch/{thu_muc}" && echo "[$(date +%H:%M:%S)] BẮT ĐẦU {ten}" && '
            f'python -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 '
            f'"{file}" > "{log}" 2>&1 && echo "[$(date +%H:%M:%S)] XONG {ten}" '
            f'|| echo "[$(date +%H:%M:%S)] LỖI {ten} (xem {log})"')


procs, logs = [], []
for i, names in enumerate(streams):
    cmd = "; ".join(lenh(n) for n in names)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
    logs.append(os.path.join(WORK, f"luong_{i}.txt"))
    procs.append(subprocess.Popen(["bash", "-c", cmd], env=env, stdout=open(logs[-1], "w"),
                                  stderr=subprocess.STDOUT))
    print(f"Luồng {i} (GPU {i}): {' -> '.join(names)}")

start = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60)
    trang_thai = " | ".join(f"luồng {i}: " + (open(l).read().strip().splitlines() or ["(chưa bắt đầu)"])[-1]
                            for i, l in enumerate(logs))
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().replace("\n", " ; ")
    print(f"{int(time.time() - start) // 60:3d} phút | GPU {gpu} | {trang_thai}")

print()
for i, l in enumerate(logs):
    print(f"===== Luồng {i} =====")
    print(open(l).read())

## 4. Kiểm tra nhanh

In `so_sanh.csv` (bảng Keras ↔ PyTorch) và đuôi log của notebook lỗi (nếu có).

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 60)
bang = pd.read_csv("results/so_sanh.csv")
display(bang[["notebook", "chi_so", "keras", "pytorch", "chenh_lech", "pytorch_params",
              "epochs_pytorch", "device", "train_time_s"]])

for ten in NOTEBOOK:
    log = os.path.join(WORK, f"log_{ten}.txt")
    if os.path.exists(log) and "Error" in open(log, encoding="utf-8", errors="replace").read():
        print(f"===== LỖI: {ten} =====")
        print(open(log, encoding="utf-8", errors="replace").read()[-3000:])

## 5. Đóng gói kết quả để tải về

Zip gồm: các notebook đã có output, `results/` (không kèm `checkpoints/`, `.pt`), các file log. Đường dẫn trong zip tính từ thư mục BTL03, nên giải nén đè thẳng vào thư mục đó trong repo.

In [ ]:
import zipfile

name = f"btl03_gpu_{time.strftime('%Y%m%d-%H%M')}.zip"
zip_path = os.path.join(WORK, name)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for ten, (thu_muc, file) in NOTEBOOK.items():
        z.write(os.path.join("pytorch", thu_muc, file))
    for root, dirs, files in os.walk("results"):
        dirs[:] = [d for d in dirs if d != "checkpoints"]
        for f in files:
            if not f.endswith((".pt", ".lock")):
                z.write(os.path.join(root, f))
    for f in glob.glob(os.path.join(WORK, "*.txt")):
        z.write(f, os.path.join("_log_kaggle", os.path.basename(f)))
print("Đã tạo", zip_path, os.path.getsize(zip_path) // 1024, "KB")
for n in sorted(zipfile.ZipFile(zip_path).namelist()):
    print("  ", n)